# Phase 2 - Hyperparameter tuning (Colab T4)

Tunes `lr0`, `box`, `cls`, `dfl` for the **hazard** detector (the one with
headroom: baseline mAP50 = 0.566 vs child's 0.947).

## Surviving the Colab GPU time limit

Free Colab will disconnect you mid-sweep sooner or later, and `runs/` lives in
the **ephemeral session** - it is wiped on disconnect. So this notebook writes
the sweep to **Google Drive** instead. That makes the tuner's built-in resume
actually usable across sessions.

If you get disconnected (or hit the GPU quota):
1. Come back later, re-run the setup cells, then re-run the **same** tuning
   cell with the **same** `--iterations`.
2. `--iterations` is a **total target**, not an increment - the tuner reads the
   existing `tune_results.csv` and runs only the remaining iterations.
3. Only the in-flight iteration is lost.

**Budget:** 6 iterations x 20 epochs is about **2.9 h** on a T4 (measured:
1.45 min/epoch). Your hazard baseline session survived 2.4 h, so this is in
proven territory - but the Drive setup means overrunning is survivable.

In [ ]:
!nvidia-smi

In [ ]:
# Pinned: the sweep + baselines ran on 8.4.106. Ablations must
# compare like with like, and 8.4.x changed head init
# ("Remapped N/12 cls head rows...") vs 8.3.x.
!pip install -q ultralytics==8.4.106 roboflow

In [ ]:
# Mount Drive so the sweep survives a disconnect
from google.colab import drive
drive.mount("/content/drive")
TUNE_PROJECT = "/content/drive/MyDrive/deeplrn_group2/runs"
import os; os.makedirs(TUNE_PROJECT, exist_ok=True)
print("sweep will be written to:", TUNE_PROJECT)

In [ ]:
import os
REPO_URL = "https://github.com/FooJames/DEEPLRN_Group2.git"
if not os.path.isdir("DEEPLRN_Group2"):
    !git clone $REPO_URL
else:
    !cd DEEPLRN_Group2 && git pull
%cd DEEPLRN_Group2

In [ ]:
from google.colab import userdata
import os
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
print("key loaded:", bool(os.environ.get("ROBOFLOW_API_KEY")))

In [ ]:
# Hazard dataset only (child isn't being tuned in this pass)
!python scripts/download_data.py --child-version 3 --hazard-version 1 --only hazard
!python scripts/fix_data_yaml.py data/hazard/data.yaml

## Tune the hazard detector

T4 budget (1.45 min/epoch measured on this dataset):

| iterations | 15 ep | 20 ep | 30 ep |
|---|---|---|---|
| 6  | 2.2 h | **2.9 h** | 4.3 h |
| 8  | 2.9 h | 3.9 h | 5.8 h |
| 10 | 3.6 h | 4.8 h | 7.2 h |

20 epochs is a *ranking* proxy, not convergence (hazard reaches ~0.46 mAP50 by
epoch 20 vs 0.53 at 100) - fine for comparing configs, and the winner is
retrained at full length in Phase 4.

`--optimizer AdamW` is deliberate: `auto` silently ignores `lr0`.

**Re-run this exact cell to resume after a disconnect.**

In [ ]:
!python scripts/tune.py --model hazard --data data/hazard/data.yaml     --iterations 6 --epochs 20 --optimizer AdamW --project "$TUNE_PROJECT"

### Progress check (safe to run anytime, even mid-sweep)

In [ ]:
# How many iterations are banked on Drive so far?
import os, pandas as pd
csv = os.path.join(TUNE_PROJECT, "tune_hazard", "tune_results.csv")
if os.path.isfile(csv):
    df = pd.read_csv(csv)
    print(f"{len(df)} / 6 iterations complete
")
    print(df.sort_values("fitness", ascending=False).to_string(index=False))
else:
    print("no iterations recorded yet")

### Save the sweep (once all iterations are done)

In [ ]:
!zip -r hazard_tuning.zip results/metrics/tuning_hazard.csv results/metrics/tuning_hazard_best.yaml "$TUNE_PROJECT/tune_hazard"
!unzip -l hazard_tuning.zip
from google.colab import files
files.download("hazard_tuning.zip")

In [ ]:
!cat results/metrics/tuning_hazard_best.yaml
print("
baseline to beat: hazard mAP50=0.5657 / mAP50-95=0.4072")

### Notes
- The sweep lives on Drive, so even without downloading the zip your results
  are safe - the zip is just a convenience copy for the repo.
- Tuning evaluates on the **val** split only; the test split stays untouched
  until the very end.
- Child tuning is intentionally skipped (0.947 baseline = little headroom).
  Same command with `--model child` if you want it later.
- Caveat for the write-up: `lr0` is tuned under AdamW, so the Phase 3 optimizer
  ablation should use per-optimizer learning rates rather than reusing this
  `lr0` for SGD.